In [11]:
import pandas as pd

In [12]:
df = pd.read_csv('monthly_electric_data.csv')

In [13]:
df.head(20)

,Area,ISO 3 code,Date,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity demand,Demand,Demand,TWh,12.77,NaN,NaN
1,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,%,34.57,NaN,NaN
2,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,%,65.44,NaN,NaN
3,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Gas and Other Fossil,%,63.40,NaN,NaN
4,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, Bioenergy and Other Renewables",%,29.08,NaN,NaN
5,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Renewables,%,29.55,NaN,NaN
6,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Wind and Solar,%,0.47,NaN,NaN
7,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,TWh,4.41,NaN,NaN
8,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,TWh,8.35,NaN,NaN
9,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Gas and Other Fossil,TWh,8.09,NaN,NaN


In [14]:
import pandas as pd
import numpy as np

# 1. Lọc đơn vị TWh và loại bỏ các nhóm/khu vực
df_twh = df[(df['Unit'] == 'TWh') &
            (df['Category'] == 'Electricity generation') &
            (df['Area type'] == 'Country or economy')]

# 2. Lọc riêng 4 nguồn năng lượng mục tiêu
df_fuel = df_twh[df_twh['Variable'].isin(['Coal', 'Hydro', 'Gas', 'Solar'])]

# Tính trung bình sản lượng mỗi loại nhiên liệu
avg_gen = df_fuel.groupby(['Area', 'Variable'])['Value'].mean().unstack().reset_index()

# Đảm bảo có mặt đủ 4 cột (Phòng trường hợp nước nào đó không có ghi nhận lượng điện loại này)
for col in ['Coal', 'Hydro', 'Gas', 'Solar']:
    if col not in avg_gen.columns:
        avg_gen[col] = 0
avg_gen.fillna(0, inplace=True)

# 3. Lấy dữ liệu mốc của Việt Nam
vn_data = avg_gen[avg_gen['Area'] == 'Viet Nam']
if not vn_data.empty:
    vn_coal = vn_data['Coal'].values[0]
    vn_hydro = vn_data['Hydro'].values[0]
    vn_gas = vn_data['Gas'].values[0]
    vn_solar = vn_data['Solar'].values[0]

    # Tính khoảng cách Euclidean trên 4 trục
    avg_gen['Độ lệch (Distance)'] = np.sqrt(
        (avg_gen['Coal'] - vn_coal)**2 +
        (avg_gen['Hydro'] - vn_hydro)**2 +
        (avg_gen['Gas'] - vn_gas)**2 +
        (avg_gen['Solar'] - vn_solar)**2
    )

    # Xếp hạng 20 nước gần giống VN nhất
    similar_countries = avg_gen.sort_values('Độ lệch (Distance)').head(21).iloc[1:]

    # Định dạng hiển thị
    similar_countries = similar_countries.rename(columns={
        'Area': 'Quốc gia',
        'Coal': 'Than (TWh)',
        'Hydro': 'Thủy điện (TWh)',
        'Gas': 'Gas (TWh)',
        'Solar': 'Mặt trời (TWh)'
    })

    pd.set_option('display.max_rows', 50)
    display(similar_countries[['Quốc gia', 'Than (TWh)', 'Thủy điện (TWh)', 'Gas (TWh)', 'Mặt trời (TWh)', 'Độ lệch (Distance)']].reset_index(drop=True))
else:
    print("Không tìm thấy dữ liệu của Việt Nam.")


Variable,Quốc gia,Than (TWh),Thủy điện (TWh),Gas (TWh),Mặt trời (TWh),Độ lệch (Distance)
0,Turkey,9.452737,5.684105,6.225579,1.439368,4.193695
1,Australia,12.913631,1.044862,1.559169,0.877477,6.815778
2,Malaysia,6.357917,2.514375,4.879583,0.132708,6.953945
3,Poland,9.334015,0.141136,0.895833,0.509015,7.531747
4,Taiwan,9.868135,0.370482,5.787717,0.248939,7.830000
5,Germany,13.882576,1.589848,6.437500,4.393333,8.138677
6,Kazakhstan,5.844255,0.822447,2.264255,0.116489,8.152454
7,Philippines (the),5.005000,0.795000,1.648077,0.170192,8.711394
8,Ukraine,4.149143,0.596667,0.767810,0.000000,9.567958
9,Pakistan,1.966220,3.139268,3.208537,0.894390,9.572440


In [15]:
import os
import pandas as pd
import numpy as np

# Giả sử bạn đã load data ban đầu
df = pd.read_csv('monthly_electric_data.csv')

# --- ĐOẠN THÊM MỚI ---
# 1. Chuyển đổi cột Date sang dạng datetime (để dễ dàng so sánh thời gian)
df['Date'] = pd.to_datetime(df['Date'])

# 2. Lọc df gốc để chỉ giữ lại dữ liệu từ đầu năm 2018 trở đi
df = df[df['Date'] >= '2018-01-01']
# ---------------------

# 3. Tính toán danh sách Top 20 quốc gia giống Việt Nam (Từ năm 2018)
df_twh = df[(df['Unit'] == 'TWh') &
            (df['Category'] == 'Electricity generation') &
            (df['Area type'] == 'Country or economy')]

df_fuel = df_twh[df_twh['Variable'].isin(['Coal', 'Hydro', 'Gas', 'Solar'])]
avg_gen = df_fuel.groupby(['Area', 'Variable'])['Value'].mean().unstack().reset_index()

for col in ['Coal', 'Hydro', 'Gas', 'Solar']:
    if col not in avg_gen.columns:
        avg_gen[col] = 0
avg_gen.fillna(0, inplace=True)

vn_data = avg_gen[avg_gen['Area'] == 'Viet Nam']
vn_coal = vn_data['Coal'].values[0]
vn_hydro = vn_data['Hydro'].values[0]
vn_gas = vn_data['Gas'].values[0]
vn_solar = vn_data['Solar'].values[0]

avg_gen['Distance'] = np.sqrt(
    (avg_gen['Coal'] - vn_coal)**2 +
    (avg_gen['Hydro'] - vn_hydro)**2 +
    (avg_gen['Gas'] - vn_gas)**2 +
    (avg_gen['Solar'] - vn_solar)**2
)

# Lấy mảng tên của Top 20 nước sát với Việt Nam nhất
top20_countries = avg_gen.sort_values('Distance').head(21)['Area'].tolist()

# 4. Lọc bộ dữ liệu gốc (đã bao gồm điều kiện từ 2018)
df_top20_data = df[df['Area'].isin(top20_countries)]

# 5. Xuất file
save_path = r"C:\Users\ADMIN\Downloads\MODEL_TFT\data\raw\ember_data\similar_20_countries_data_from2018.csv"

# Lưu file dạng csv
df_top20_data.to_csv(save_path, index=False)

print(f"✅ Đã lưu thành công dữ liệu từ năm 2018 trở đi của {len(top20_countries)} quốc gia!")
print(f"Dữ liệu được lưu tại:\n👉 {save_path}")
print(f"Tổng số dòng của Dataset mới: {len(df_top20_data):,} dòng.")


✅ Đã lưu thành công dữ liệu từ năm 2018 trở đi của 21 quốc gia!
Dữ liệu được lưu tại:
👉 C:\Users\ADMIN\Downloads\MODEL_TFT\data\raw\ember_data\similar_20_countries_data_from2018.csv
Tổng số dòng của Dataset mới: 87,880 dòng.
